In [ ]:
!pip install mne pyedflib

In [ ]:
#!/usr/bin/env python3
"""
Sleep Wearable Intelligent Agent
M.Tech Assignment: Explainable age-adaptive sleep/wellbeing agent.

The program downloads a real Sleep-EDF hypnogram, extracts sleep stages,
computes Light/Deep/REM proportions, applies fuzzy reasoning, and generates
an explainable sleep score plus a non-clinical sleep-associated wellbeing proxy.
"""

from __future__ import annotations

import argparse
import re
import urllib.request
from dataclasses import dataclass, asdict
from pathlib import Path

import pandas as pd

BASE_URL = "https://physionet.org/files/sleep-edfx/1.0.0/sleep-cassette/"
DEFAULT_RECORD = "SC4002EC-Hypnogram.edf"


@dataclass
class SleepMetrics:
    age: int
    stress_level: float
    wake_min: float
    light_min: float
    deep_min: float
    rem_min: float
    total_sleep_min: float
    total_sleep_hours: float
    light_pct: float
    deep_pct: float
    rem_pct: float
    sleep_score: float
    wellbeing_score: float
    sleep_category: str
    wellbeing_category: str


def trapezoid_membership(x: float, a: float, b: float,
                         c: float, d: float) -> float:
    """Fuzzy membership in the range 0..100."""
    if x <= a or x >= d:
        return 0.0
    if b <= x <= c:
        return 100.0
    if a < x < b:
        return 100.0 * (x - a) / (b - a)
    return 100.0 * (d - x) / (d - c)


def duration_membership(age: int, hours: float) -> float:
    """Age-adaptive sleep-duration membership."""
    if age < 18:
        return trapezoid_membership(hours, 6.5, 8.0, 10.0, 11.0)
    if age < 65:
        return trapezoid_membership(hours, 5.5, 7.0, 9.0, 10.5)
    return trapezoid_membership(hours, 5.5, 7.0, 8.0, 9.5)


def fuzzy_sleep_score(age: int, hours: float, light_pct: float,
                      deep_pct: float, rem_pct: float) -> dict:
    duration_score = duration_membership(age, hours)
    rem_score = trapezoid_membership(rem_pct, 10, 20, 25, 35)
    deep_score = trapezoid_membership(deep_pct, 5, 13, 23, 35)
    light_score = trapezoid_membership(light_pct, 30, 50, 60, 75)

    score = (
        0.30 * duration_score +
        0.25 * rem_score +
        0.25 * deep_score +
        0.20 * light_score
    )

    return {
        "duration": duration_score,
        "rem": rem_score,
        "deep": deep_score,
        "light": light_score,
        "sleep_score": score,
    }


def sleep_category(score: float) -> str:
    if score >= 85:
        return "Excellent"
    if score >= 70:
        return "Good / needs minor improvement"
    if score >= 50:
        return "Fair / needs improvement"
    return "Poor / needs attention"


def wellbeing_indicator(sleep_score: float, stress_level: float) -> float:
    """
    Academic proxy only.
    Lower self-reported stress contributes more positively.
    """
    stress_score = 100.0 - ((stress_level - 1.0) / 9.0) * 100.0
    return 0.75 * sleep_score + 0.25 * stress_score


def wellbeing_category(score: float) -> str:
    if score >= 85:
        return "Higher sleep-associated wellbeing"
    if score >= 70:
        return "Moderate-to-good sleep-associated wellbeing"
    if score >= 50:
        return "Needs attention"
    return "High concern; consider professional guidance"


def parse_stage_description(description: str) -> str | None:
    """Normalize common Sleep-EDF EDF+ annotation descriptions."""
    d = str(description).strip().lower()

    patterns = [
        (r"\bsleep stage w\b|\bwake\b|\bstage w\b", "W"),
        (r"\bsleep stage 1\b|\bstage 1\b|\bn1\b", "N1"),
        (r"\bsleep stage 2\b|\bstage 2\b|\bn2\b", "N2"),
        (r"\bsleep stage 3\b|\bstage 3\b|\bn3\b", "N3"),
        (r"\bsleep stage 4\b|\bstage 4\b|\bn4\b", "N4"),
        (r"\bsleep stage r\b|\brem\b", "REM"),
        (r"\bmovement\b|\bstage m\b|\bm\b", "M"),
    ]

    for pattern, label in patterns:
        if re.search(pattern, d):
            return label
    return None


def download_hypnogram(record: str, data_dir: Path) -> Path:
    data_dir.mkdir(parents=True, exist_ok=True)
    path = data_dir / record

    if path.exists() and path.stat().st_size > 0:
        return path

    url = BASE_URL + record
    print(f"Downloading: {url}")
    urllib.request.urlretrieve(url, path)
    return path


def read_hypnogram(path: Path) -> pd.DataFrame:
    try:
        import mne
    except ImportError as exc:
        raise RuntimeError(
            "MNE is required. Install with: pip install mne pyedflib"
        ) from exc

    annotations = mne.read_annotations(str(path))
    rows = []

    for onset, duration, description in zip(
        annotations.onset,
        annotations.duration,
        annotations.description
    ):
        stage = parse_stage_description(description)
        if stage is not None:
            rows.append({
                "onset_sec": float(onset),
                "duration_sec": float(duration),
                "stage": stage,
            })

    if not rows:
        raise ValueError("No sleep-stage annotations found.")

    return pd.DataFrame(rows)


def summarize(df: pd.DataFrame, age: int, stress_level: float) -> SleepMetrics:
    stage_minutes = (
        df.groupby("stage")["duration_sec"].sum().div(60.0).to_dict()
    )

    wake = stage_minutes.get("W", 0.0)
    n1 = stage_minutes.get("N1", 0.0)
    n2 = stage_minutes.get("N2", 0.0)
    n3 = stage_minutes.get("N3", 0.0)
    n4 = stage_minutes.get("N4", 0.0)
    rem = stage_minutes.get("REM", 0.0)

    light = n1 + n2
    deep = n3 + n4
    total_sleep = light + deep + rem

    if total_sleep <= 0:
        raise ValueError("Total scored sleep is zero.")

    light_pct = 100.0 * light / total_sleep
    deep_pct = 100.0 * deep / total_sleep
    rem_pct = 100.0 * rem / total_sleep
    hours = total_sleep / 60.0

    fuzzy = fuzzy_sleep_score(
        age, hours, light_pct, deep_pct, rem_pct
    )
    score = fuzzy["sleep_score"]
    wellbeing = wellbeing_indicator(score, stress_level)

    return SleepMetrics(
        age=age,
        stress_level=stress_level,
        wake_min=wake,
        light_min=light,
        deep_min=deep,
        rem_min=rem,
        total_sleep_min=total_sleep,
        total_sleep_hours=hours,
        light_pct=light_pct,
        deep_pct=deep_pct,
        rem_pct=rem_pct,
        sleep_score=score,
        wellbeing_score=wellbeing,
        sleep_category=sleep_category(score),
        wellbeing_category=wellbeing_category(wellbeing),
    )


def explain(metrics: SleepMetrics) -> list[str]:
    reasons = []

    if metrics.total_sleep_hours < 7:
        reasons.append(
            "Total sleep is below the usual adult target range; improving "
            "sleep duration may help."
        )
    elif metrics.total_sleep_hours > 9:
        reasons.append(
            "Sleep duration is above the usual adult target range; interpret "
            "this together with age and context."
        )
    else:
        reasons.append(
            "Sleep duration is within the usual adult target range for this age."
        )

    if 20 <= metrics.rem_pct <= 25:
        reasons.append(
            "REM proportion is close to the fuzzy agent's preferred range."
        )
    elif metrics.rem_pct < 20:
        reasons.append(
            "REM proportion is lower than the preferred range."
        )
    else:
        reasons.append(
            "REM proportion is above the preferred range."
        )

    if 13 <= metrics.deep_pct <= 23:
        reasons.append(
            "Deep-sleep proportion is within the preferred range."
        )
    elif metrics.deep_pct < 13:
        reasons.append(
            "Deep sleep is below the preferred range."
        )
    else:
        reasons.append(
            "Deep sleep is above the preferred range."
        )

    if metrics.stress_level >= 7:
        reasons.append(
            "Reported stress is high and lowers the wellbeing proxy."
        )
    elif metrics.stress_level >= 4:
        reasons.append(
            "Reported stress is moderate and has a noticeable effect on "
            "the wellbeing proxy."
        )
    else:
        reasons.append(
            "Reported stress is relatively low."
        )

    return reasons


def print_report(metrics: SleepMetrics, source: str):
    print("\n" + "=" * 72)
    print("        SLEEP WEARABLE INTELLIGENT AGENT       ")
    print("=" * 72)
    print(f"Data source            : {source}")
    print(f"Age                    : {metrics.age} years")
    print(f"Input stress level     : {metrics.stress_level:.1f}/10")
    print("-" * 72)
    print(
        f"Wake                   : {metrics.wake_min:8.2f} min"
    )
    print(
        f"Light sleep (N1+N2)   : {metrics.light_min:8.2f} min "
        f"({metrics.light_pct:5.2f}%)"
    )
    print(
        f"Deep sleep (N3+N4)    : {metrics.deep_min:8.2f} min "
        f"({metrics.deep_pct:5.2f}%)"
    )
    print(
        f"REM sleep             : {metrics.rem_min:8.2f} min "
        f"({metrics.rem_pct:5.2f}%)"
    )
    print(
        f"Total scored sleep    : {metrics.total_sleep_min:8.2f} min "
        f"({metrics.total_sleep_hours:.2f} h)"
    )
    print("-" * 72)
    print(f"Sleep Score            : {metrics.sleep_score:6.2f}/100")
    print(f"Sleep Assessment       : {metrics.sleep_category}")
    print(f"Wellbeing Proxy        : {metrics.wellbeing_score:6.2f}/100")
    print(f"Wellbeing Assessment   : {metrics.wellbeing_category}")
    print("-" * 72)
    print("Explainable reasoning:")
    for i, reason in enumerate(explain(metrics), 1):
        print(f"  {i}. {reason}")
    print("-" * 72)
    print("Agent recommendation:")
    if metrics.sleep_score >= 85:
        print(
            "  Maintain the current sleep routine and monitor trends "
            "across nights."
        )
    elif metrics.sleep_score >= 70:
        print(
            "  Maintain adequate sleep time and improve the weakest "
            "sleep-stage component."
        )
    elif metrics.sleep_score >= 50:
        print(
            "  Work on sleep regularity, sleep duration and stress reduction."
        )
    else:
        print(
            "  Persistent poor results should be discussed with a qualified "
            "professional."
        )
    print(
        "NOTE: The wellbeing value is an academic proxy, not a clinical "
        "mental-health diagnosis."
    )
    print("=" * 72)


def save_result(metrics: SleepMetrics, output_path: Path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([asdict(metrics)]).to_csv(
        output_path, index=False
    )


def main():
    parser = argparse.ArgumentParser(
        description="Explainable Sleep Wearable Intelligent Agent"
    )
    parser.add_argument(
        "--record",
        default=DEFAULT_RECORD,
        help="Sleep-EDF hypnogram filename"
    )
    parser.add_argument(
        "--age",
        type=int,
        default=33,
        help="Person age in years"
    )
    parser.add_argument(
        "--stress",
        type=float,
        default=4.0,
        help="Self-reported stress level from 1 to 10"
    )
    parser.add_argument(
        "--data-dir",
        default="data",
        help="Dataset directory"
    )
    parser.add_argument(
        "--output",
        default="results/sleep_agent_result.csv",
        help="Output CSV"
    )
    args, unknown = parser.parse_known_args()

    if not 1 <= args.stress <= 10:
        raise ValueError("Stress must be between 1 and 10.")
    if not 1 <= args.age <= 120:
        raise ValueError("Age must be between 1 and 120.")

    path = download_hypnogram(
        args.record, Path(args.data_dir)
    )
    df = read_hypnogram(path)
    metrics = summarize(df, args.age, args.stress)

    print_report(
        metrics,
        f"PhysioNet Sleep-EDF ({args.record})"
    )
    save_result(
        metrics,
        Path(args.output)
    )


if __name__ == "__main__":
    main()



        SLEEP WEARABLE INTELLIGENT AGENT       
Data source            : PhysioNet Sleep-EDF (SC4002EC-Hypnogram.edf)
Age                    : 33 years
Input stress level     : 4.0/10
------------------------------------------------------------------------
Wake                   :   942.50 min
Light sleep (N1+N2)   :   216.00 min (45.76%)
Deep sleep (N3+N4)    :   148.50 min (31.46%)
REM sleep             :   107.50 min (22.78%)
Total scored sleep    :   472.00 min (7.87 h)
------------------------------------------------------------------------
Sleep Score            :  78.13/100
Sleep Assessment       : Good / needs minor improvement
Wellbeing Proxy        :  75.27/100
Wellbeing Assessment   : Moderate-to-good sleep-associated wellbeing
------------------------------------------------------------------------
Explainable reasoning:
  1. Sleep duration is within the usual adult target range for this age.
  2. REM proportion is close to the fuzzy agent's preferred range.
  3. Deep slee